# BBQ + TransformerLens activation probes

Quick exploration notebook:
1. Load the BBQ dataset.
2. Load a HookedTransformer model.
3. Pull per-layer residual-stream activations.
4. Train linear probes and plot accuracy across depth.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from mech_interp_bbq.data import load_bbq, to_examples, BBQ_CATEGORIES
from mech_interp_bbq.activations import load_model, collect_resid_post
from mech_interp_bbq.probes import train_all_layers

In [ ]:
ds = load_bbq(category="Gender_identity", max_examples=200)
examples = to_examples(ds)
print(f"Loaded {len(examples)} examples")
print(examples[0].prompt())

In [ ]:
model = load_model("gpt2-small")
model.cfg

In [ ]:
prompts = [ex.prompt() for ex in examples]
labels = np.array([ex.label for ex in examples])
batch = collect_resid_post(model, prompts, batch_size=8)
batch.acts.shape  # (n, n_layers, d_model)

In [ ]:
results = train_all_layers(batch.acts, labels)
accs = [r.mean_accuracy for r in results]
plt.figure(figsize=(7, 4))
plt.plot(range(len(accs)), accs, marker="o")
plt.xlabel("layer")
plt.ylabel("5-fold CV accuracy")
plt.title("Per-layer linear probe (BBQ label)")
plt.grid(alpha=0.3)
plt.show()